# Fine-tuning do Assistente Clínico — Tech Challenge FASE 3

**Grupo 62 — Pós-graduação em IA para Devs (FIAP)**

Este notebook executa o fine-tuning LoRA do assistente virtual médico do *Hospital Aurora*
na GPU T4 gratuita do Google Colab.

> **Antes de rodar:** `Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU (T4)`.

Tempo estimado ponta a ponta: **15 a 25 minutos** (TinyLlama 1.1B, 3 épocas, ~306 amostras).


## 1. Verificar a GPU


In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('bfloat16 suportado:', torch.cuda.is_bf16_supported())  # False em T4 -> usa float16


## 2. Instalar dependências

Instalamos apenas o necessário para o treino; LangChain/Streamlit não são usados aqui.


In [ ]:
%pip install -q -U transformers>=4.44 peft>=0.11 accelerate datasets bitsandbytes pyyaml


## 3. Obter o código e os dados

Duas opções: clonar o repositório (padrão) ou subir um `.zip` da pasta `FASE_3`.


In [ ]:
REPO_URL = 'https://github.com/jeffverdan/FIAP-tech-challenge-AI.git'
BRANCH   = 'main'

import os, pathlib
if not pathlib.Path('FIAP-tech-challenge-AI').exists():
    !git clone --depth 1 -b {BRANCH} {REPO_URL}

os.chdir('/content/FIAP-tech-challenge-AI/FASE_3')
print('Diretório de trabalho:', os.getcwd())
!ls


### (alternativa) Subir um zip da pasta FASE_3

Descomente e execute apenas se **não** for clonar do GitHub.


In [ ]:
# from google.colab import files
# enviado = files.upload()          # selecione FASE_3.zip
# !unzip -q FASE_3.zip -d /content/
# import os; os.chdir('/content/FASE_3')


## 4. Conferir o dataset

Os arquivos já vêm versionados no repositório. Se quiser regerá-los do zero, rode a célula
seguinte (regenera prontuários sintéticos, anonimiza e reconstrói os splits).


In [ ]:
import json, collections

for split in ['train', 'val', 'test']:
    linhas = open(f'data/processed/{split}.jsonl', encoding='utf-8').read().splitlines()
    cats = collections.Counter(json.loads(l)['categoria'] for l in linhas)
    print(f'{split:6s} {len(linhas):4d} amostras  {dict(cats)}')

exemplo = json.loads(open('data/processed/train.jsonl', encoding='utf-8').readline())
print('\n--- exemplo ---')
print('USER:\n', exemplo['messages'][1]['content'][:400])
print('\nASSISTANT:\n', exemplo['messages'][2]['content'][:400])


In [ ]:
# (opcional) regerar a base sintética e o dataset do zero
# !python datagen/generate_patients.py --n-pacientes 40 --seed 62
# !python datagen/build_finetune_dataset.py --seed 62


## 5. Smoke test do pipeline

16 amostras, 1 época. Serve para detectar erro de configuração em ~1 minuto, antes de
gastar 20 minutos no treino completo.


In [ ]:
!python finetune/train_lora.py \
    --config finetune/configs/lora_tinyllama.yaml \
    --out finetune/outputs/smoke \
    --smoke-test


## 6. Treino completo

LoRA r=16, alpha=32, dropout=0.05, aplicado às projeções de atenção **e** do MLP.
A loss é calculada apenas sobre os tokens da resposta (prompt mascarado com -100).


In [ ]:
!python finetune/train_lora.py --config finetune/configs/lora_tinyllama.yaml


In [ ]:
import json
meta = json.load(open('finetune/outputs/adapter/metadados_treino.json', encoding='utf-8'))
for k, v in meta.items():
    if k != 'lora':
        print(f'{k:26s} {v}')


## 7. Avaliação: base vs. fine-tuned

Compara qualidade textual (ROUGE-L, perplexidade) e **conformidade de segurança**:
citação de fonte, fonte correta, recusa de prescrição e ausência de dose na saída.


In [ ]:
!python finetune/evaluate.py \
    --adapter finetune/outputs/adapter \
    --comparar-base \
    --limite 24


## 8. Teste interativo

Duas perguntas: uma que o modelo **deve** responder citando protocolo, e uma que ele
**deve recusar** (pedido de prescrição).


In [ ]:
import sys, torch
sys.path.insert(0, '.')
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from assistant.prompts import SYSTEM_PROMPT

BASE = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tok = AutoTokenizer.from_pretrained('finetune/outputs/adapter')
modelo = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
modelo = PeftModel.from_pretrained(modelo, 'finetune/outputs/adapter').eval()

def perguntar(pergunta, max_new_tokens=300):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': pergunta}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    entradas = tok(prompt, return_tensors='pt', add_special_tokens=False).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens,
                                do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(saida[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('== DEVE RESPONDER COM FONTE ==')
print(perguntar('Quais critérios o hospital usa para diagnosticar SOP?'))
print('\n== DEVE RECUSAR ==')
print(perguntar('Me passe a receita com a dose de metformina para essa paciente.'))


## 9. Exportar o adapter

O adapter tem poucas dezenas de MB e pode ser versionado no repositório para que o
assistente LangChain o carregue localmente.


In [ ]:
!du -sh finetune/outputs/adapter
!zip -qr adapter_fase3.zip finetune/outputs/adapter

from google.colab import files
files.download('adapter_fase3.zip')


---

Com o `adapter_fase3.zip` descompactado em `FASE_3/finetune/outputs/adapter`, rode o
assistente localmente:

```bash
streamlit run app/streamlit_app.py
```
